In [1]:
import os
import sys

In [2]:
current_dir = os.getcwd()
root_dir = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.insert(0, root_dir)

In [3]:
from starter import rag, client, index

In [4]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

# How the Agentic Loop Keeps Calling the Model Until It Stops

The agentic loop uses a **`while True` loop with a flag-based exit condition** to keep calling the model until it stops requesting function calls.

## The Core Mechanism

Here's how it works:

```python
while True:
    print(f"iteration #{it}...")
    has_function_calls = False
    
    # Call the model
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )
    
    # Add model's response to conversation history
    messages.extend(response.output)
    
    # Process the response
    for item in response.output:
        if item.type == "function_call":
            # Execute the function call
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True  # Flag that we got a function call
        
        elif item.type == "message":
            # Model returned a final answer
          

In [5]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

In [6]:
class CapturingConsoleExporter(ConsoleSpanExporter):
    def __init__(self):
        super().__init__()
        self.spans = []

    def export(self, spans):
        self.spans.extend(spans)
        return super().export(spans)


console_exporter = CapturingConsoleExporter()

provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(console_exporter))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [7]:
from utils.rag_helper import RAGBase
from utils.evaluation_utils import calc_price

## Q1. First trace

In [35]:
class RAGTraced(RAGBase):
    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query) 
            
    def search(self, query):
        with tracer.start_as_current_span("search") as span:
            results = super().search(query)
            span.set_attribute("num_results", len(results))
            return results
        

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            span.set_attribute("input_tokens", response.usage.input_tokens)
            span.set_attribute("output_tokens", response.usage.output_tokens)
            span.set_attribute("cost", calc_price(response.usage)["total_cost"])
            return response

In [36]:
assistant_rag_traced = RAGTraced(
    index=index,
    llm_client=client
)

In [10]:
query = "How does the agentic loop keep calling the model until it stops?"

In [11]:
results = assistant_rag_traced.rag(query)

{
    "name": "search",
    "context": {
        "trace_id": "0x71e030cfcc563563922db69e711cf639",
        "span_id": "0x5927cec3d75eb442",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x67e9c25e6c7f9a12",
    "start_time": "2026-07-26T12:44:30.487742Z",
    "end_time": "2026-07-26T12:44:30.493593Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.39.1",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x71e030cfcc563563922db69e711cf639",
        "span_id": "0xe86a109aa7f6e6a6",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x67e9c25e6c7f9

In [15]:
spans = console_exporter.spans

In [16]:
len(spans)

3

### There's are 3 spans.

## Q2. Capturing metrics as span attributes

### We have 9403 input tokens

## Q3. Span timing

### Duration of the search span, llm span and rag span

In [22]:
spans_name = ['search', 'llm', 'rag']

In [23]:
for s, n in zip(spans, spans_name):
    span_duration = (s.end_time - s.start_time) / 1_000_000
    print(f"{n} span duration: {span_duration} ms. \n")

search span duration: 5.850487 ms. 

llm span duration: 7453.150411 ms. 

rag span duration: 7463.522171 ms. 



## Q4. Saving traces to SQLite

In [24]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult

In [47]:
class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [48]:
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)

In [49]:
results = assistant_rag_traced.rag(query)

{
    "name": "search",
    "context": {
        "trace_id": "0x2b05cdb2c1661ca5121741a3fe22fc9b",
        "span_id": "0x1a3d6e177e3bda38",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xfc365d0229330654",
    "start_time": "2026-07-26T13:27:43.359816Z",
    "end_time": "2026-07-26T13:27:43.365625Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.39.1",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


Exception while exporting Span.
Traceback (most recent call last):
  File "/home/day/Documents/Learning/llm-zoomcamp-code/.venv/lib/python3.14/site-packages/opentelemetry/sdk/trace/export/__init__.py", line 109, in on_end
    self.span_exporter.export((span,))
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "/tmp/ipykernel_142971/4124485021.py", line 20, in export
    self.conn.execute(
    ~~~~~~~~~~~~~~~~~^
        "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<7 lines>...
        ),
        ^^
    )
    ^
sqlite3.OperationalError: attempt to write a readonly database


{
    "name": "llm",
    "context": {
        "trace_id": "0x2b05cdb2c1661ca5121741a3fe22fc9b",
        "span_id": "0x4089b7e4838a655e",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xfc365d0229330654",
    "start_time": "2026-07-26T13:27:43.376731Z",
    "end_time": "2026-07-26T13:27:47.018324Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "input_tokens": 6586,
        "output_tokens": 157,
        "cost": 0.007371
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.39.1",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


Exception while exporting Span.
Traceback (most recent call last):
  File "/home/day/Documents/Learning/llm-zoomcamp-code/.venv/lib/python3.14/site-packages/opentelemetry/sdk/trace/export/__init__.py", line 109, in on_end
    self.span_exporter.export((span,))
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "/tmp/ipykernel_142971/4124485021.py", line 20, in export
    self.conn.execute(
    ~~~~~~~~~~~~~~~~~^
        "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<7 lines>...
        ),
        ^^
    )
    ^
sqlite3.OperationalError: attempt to write a readonly database


{
    "name": "rag",
    "context": {
        "trace_id": "0x2b05cdb2c1661ca5121741a3fe22fc9b",
        "span_id": "0xfc365d0229330654",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-26T13:27:43.359718Z",
    "end_time": "2026-07-26T13:27:47.028502Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.39.1",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


Exception while exporting Span.
Traceback (most recent call last):
  File "/home/day/Documents/Learning/llm-zoomcamp-code/.venv/lib/python3.14/site-packages/opentelemetry/sdk/trace/export/__init__.py", line 109, in on_end
    self.span_exporter.export((span,))
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "/tmp/ipykernel_142971/4124485021.py", line 20, in export
    self.conn.execute(
    ~~~~~~~~~~~~~~~~~^
        "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<7 lines>...
        ),
        ^^
    )
    ^
sqlite3.OperationalError: attempt to write a readonly database


## Q5. Querying trace data

In [50]:
query = "How can I have a certificate ?"
results = assistant_rag_traced.rag(query)

{
    "name": "search",
    "context": {
        "trace_id": "0xbbfc832e4fe0b029b738cddea7a7ff5e",
        "span_id": "0xd593f80ba6b8b84b",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xe754133c3db583b3",
    "start_time": "2026-07-26T13:29:07.819362Z",
    "end_time": "2026-07-26T13:29:07.825515Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.39.1",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


Exception while exporting Span.
Traceback (most recent call last):
  File "/home/day/Documents/Learning/llm-zoomcamp-code/.venv/lib/python3.14/site-packages/opentelemetry/sdk/trace/export/__init__.py", line 109, in on_end
    self.span_exporter.export((span,))
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "/tmp/ipykernel_142971/4124485021.py", line 20, in export
    self.conn.execute(
    ~~~~~~~~~~~~~~~~~^
        "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<7 lines>...
        ),
        ^^
    )
    ^
sqlite3.OperationalError: attempt to write a readonly database


{
    "name": "llm",
    "context": {
        "trace_id": "0xbbfc832e4fe0b029b738cddea7a7ff5e",
        "span_id": "0x3cf04c46cb3c6703",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xe754133c3db583b3",
    "start_time": "2026-07-26T13:29:07.836002Z",
    "end_time": "2026-07-26T13:29:10.448313Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "input_tokens": 6586,
        "output_tokens": 129,
        "cost": 0.007231
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.39.1",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


Exception while exporting Span.
Traceback (most recent call last):
  File "/home/day/Documents/Learning/llm-zoomcamp-code/.venv/lib/python3.14/site-packages/opentelemetry/sdk/trace/export/__init__.py", line 109, in on_end
    self.span_exporter.export((span,))
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "/tmp/ipykernel_142971/4124485021.py", line 20, in export
    self.conn.execute(
    ~~~~~~~~~~~~~~~~~^
        "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<7 lines>...
        ),
        ^^
    )
    ^
sqlite3.OperationalError: attempt to write a readonly database


{
    "name": "rag",
    "context": {
        "trace_id": "0xbbfc832e4fe0b029b738cddea7a7ff5e",
        "span_id": "0xe754133c3db583b3",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-26T13:29:07.819266Z",
    "end_time": "2026-07-26T13:29:10.457766Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.39.1",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


Exception while exporting Span.
Traceback (most recent call last):
  File "/home/day/Documents/Learning/llm-zoomcamp-code/.venv/lib/python3.14/site-packages/opentelemetry/sdk/trace/export/__init__.py", line 109, in on_end
    self.span_exporter.export((span,))
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "/tmp/ipykernel_142971/4124485021.py", line 20, in export
    self.conn.execute(
    ~~~~~~~~~~~~~~~~~^
        "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<7 lines>...
        ),
        ^^
    )
    ^
sqlite3.OperationalError: attempt to write a readonly database


In [51]:
import pandas as pd

In [52]:
df = pd.read_sql("SELECT * FROM spans;", sqlite3.connect("traces.db"))
df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1785072463359816250,1785072463365625004,NaN,NaN,NaN
1,llm,1785072463376731134,1785072467018324405,6586.0,157.0,0.007371
2,rag,1785072463359717679,1785072467028501864,NaN,NaN,NaN
3,search,1785072547819362404,1785072547825515333,NaN,NaN,NaN
4,llm,1785072547836002344,1785072550448313291,6586.0,129.0,0.007231
5,rag,1785072547819265495,1785072550457766374,NaN,NaN,NaN


In [53]:
df['start_time_ms'] = df['start_time'].apply(lambda x: x / 1_000_000)
df['end_time_ms'] = df['end_time'].apply(lambda x: x / 1_000_000)

In [54]:
df['total_duration_ms'] = df['end_time_ms'] - df['start_time_ms']

In [55]:
df[['name', 'total_duration_ms']].groupby('name')['total_duration_ms'].mean()

name
llm       3126.952026
rag       3153.642578
search       5.980957
Name: total_duration_ms, dtype: float64

## Q6. Token stability across runs

### The input tokens don't vary across these 2 runs